# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a dataset defined by a [Croissant schema](https://mlcommons.org/croissant/) using the `mlcroissant` Python library.

### Dataset Source
We use the FAIR\^2 dataset, loaded directly from its Croissant schema URL hosted by SEN Science.


In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")

### Dataset context, coverage, and notes
- **Identifier:** 10.71728/senscience.y7m0-f273
- **License:** https://opendatacommons.org/licenses/by/1-0/
- **Spatial coverage:** Samburu, Isiolo, Marsabit counties, Northern Kenya
- **Temporal coverage:** 2021-11-16 to 2024-11-16

This dataset comprises survey records and ordered logistic regression outputs for analyzing factors affecting knowledge adoption in rangeland management.

## 2. Data Overview
### Review all available Record Sets, Fields, and their `@id`s.
We'll inspect the dataset's available record sets (tables/sheets), their fields, and field `@id` identifiers. All references will use `@id`, in line with Croissant URIs.

In [ ]:
# List all available record sets and their IDs
print("Available record sets in the dataset:")
record_sets = list(dataset.list_record_sets())  # Each is a dict with metadata
for rs_meta in record_sets:
    rs_id = rs_meta['@id'] if '@id' in rs_meta else '(no @id)'
    rs_name = rs_meta.get('name', '[No name]')
    print(f" - Name: {rs_name}\n   @id: {rs_id}")

# For each record set, list all fields and their @id values
for rs_meta in record_sets:
    rs_id = rs_meta['@id']
    print(f"\nRecord Set: {rs_meta.get('name','[No name]')} (@id: {rs_id})")
    try:
        fields = dataset.list_fields(record_set=rs_id)
        if fields:
            for field in fields:
                name = field.get('name', '[No name]')
                field_id = field.get('@id', '[No @id]')
                datatype = field.get('dataType', '[No dataType]')
                print(f"   - Field name: {name}\n     @id: {field_id}\n     dataType: {datatype}")
        else:
            print("   (No fields found)")
    except Exception as e:
        print(f"   (Could not list fields: {e})")

**Note:** The structure above is critical for referencing any piece of data for analysis by its `@id` (not by order or name).

## 3. Data Extraction
We'll load all records from each record set into pandas DataFrames, referencing record sets by their `@id`.

In [ ]:
# Gather all record set @ids to load
record_set_ids = [rs['@id'] for rs in dataset.list_record_sets()]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from record set @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f" - Loaded {len(df)} rows. Columns: {list(df.columns)}")
    except Exception as e:
        print(f" - Error loading record set {record_set_id}: {e}")

# As an example, show columns and preview the first rows for the first available record set
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nFirst record set: {first_rs_id}")
    print(f"Available columns: {dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field by its `@id` from the first available record set for basic analysis: filter, normalize, and group. All references use entity `@id` fields only.

In [ ]:
# Example EDA: Filter, normalize, and group on a numeric field identified by its @id
from IPython.display import display

# Let's use the first record set and the first numeric field found
record_set_id = None
numeric_field_id = None
group_field_id = None

# Find the first record set with at least one numeric field
for rs in dataset.list_record_sets():
    rs_id = rs['@id']
    fields = dataset.list_fields(record_set=rs_id)
    for field in fields:
        # Check for Croissant numeric data types
        dt = field.get('dataType', '').lower()
        if dt in ['number', 'float', 'integer', 'http://schema.org/number', 'http://schema.org/float', 'http://schema.org/integer']:
            record_set_id = rs_id
            numeric_field_id = field['@id']
            break
    if record_set_id and numeric_field_id:
        # Optionally find a groupable field as example
        for field in fields:
            if field['@id'] != numeric_field_id:
                # Use first non-numeric for group
                group_field_id = field['@id']
                break
        break

if not (record_set_id and numeric_field_id):
    raise ValueError("No suitable record set with a numeric field was found.")

df = dataframes[record_set_id]

# Show field and group info
print(f"Using record_set_id: {record_set_id}")
print(f"Using numeric_field_id: {numeric_field_id}")
if group_field_id:
    print(f"Using group_field_id: {group_field_id}")

# Filter numeric field (example: keep values above mean)
if numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else None
    if threshold is not None:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Optional group by group_field_id
        if group_field_id and (group_field_id in filtered_df.columns):
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_'+numeric_field_id)
            print(f"Mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    else:
        print(f"Field {numeric_field_id} is not numeric.")
else:
    print(f"Column {numeric_field_id} not found in DataFrame.")

## 5. Visualization
We'll visualize the distribution of the chosen numeric field and optionally relationships to the group field, referencing all by their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric_field_id
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f'Distribution of field {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.show()

# If group_field_id exists, plot mean by group
if group_field_id and group_field_id in df.columns:
    grouped = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    plt.figure(figsize=(9,4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped)
    plt.title(f'Mean {numeric_field_id} per {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- This notebook demonstrated step-by-step exploration of a Croissant-based dataset using the `mlcroissant` library.
- All data elements are referenced exclusively by their Croissant `@id`, ensuring robust, portable code.
- We've loaded metadata, inspected schema structure, extracted records, performed basic EDA, and visualized results.

You can extend this analysis with advanced filtering, feature engineering, and further visualizations using the powerful combination of Croissant schemas and the Python data ecosystem.